# 🎬 EscenIA · Generador de imágenes con IA (gratis en Google Colab)

**Hecho para tu canal de videos.** Este cuaderno usa una tarjeta gráfica potente que Google te presta **gratis** para crear imágenes estilo animación 3D de cine.

### Cómo usarlo (muy fácil)
1. Arriba en el menú: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador por hardware: GPU (T4) → Guardar**.
2. Ejecuta cada celda en orden tocando el botón ▶ de la izquierda. Espera a que termine (sale un ✓).
3. En la **última celda** escribe tu historia y tu personaje, y ejecútala. Se generan todas las imágenes y se descargan en un archivo .zip.

> La primera vez, la celda 2 tarda unos minutos en preparar el modelo. Es normal.


## Celda 1 · Preparar el programa
Ejecuta esta celda. Instala lo necesario.

In [ ]:
# Instala las librerías (1-2 minutos)
!pip -q install diffusers transformers accelerate safetensors --upgrade
import torch
print("GPU detectada:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU — activa la GPU en Entorno de ejecución")


## Celda 2 · Cargar el modelo de imágenes
Ejecuta y espera a que diga **Modelo listo**. (La primera vez descarga ~7 GB, tarda unos minutos.)

In [ ]:
from diffusers import StableDiffusionXLPipeline
import torch

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16, variant="fp16", use_safetensors=True
).to("cuda")
pipe.enable_vae_tiling()
try:
    pipe.enable_xformers_memory_efficient_attention()
except Exception:
    pass
print("✅ Modelo listo")


## Celda 3 · Crear tu video (imágenes)
Escribe tu historia y tu personaje, elige estilo y formato, y ejecuta.
El cuaderno divide la historia en escenas, genera una imagen por escena **manteniendo tu personaje**, las muestra y te las descarga en un .zip.


In [ ]:
#@title ✍️ Escribe aquí y pulsa ▶ { display-mode: "form" }

historia = "Hab\u00eda una vez un peque\u00f1o dinosaurio llamado Rex. Viv\u00eda en un valle lleno de flores brillantes y cascadas de colores. Un d\u00eda, Rex descubri\u00f3 una cueva misteriosa escondida entre las rocas. Dentro brillaba un cristal m\u00e1gico con todos los colores del arco\u00edris. Rex supo que esa noche comenzar\u00eda la aventura m\u00e1s grande de su vida." #@param {type:"string"}
personaje = "un peque\u00f1o dinosaurio verde de ojos grandes y amigables" #@param {type:"string"}
estilo = "Animaci\u00f3n 3D (Pixar)" #@param ["Animaci\u00f3n 3D (Pixar)", "Anime / Ghibli", "Acuarela infantil", "Realista cine", "C\u00f3mic"]
formato = "Vertical (Shorts/TikTok)" #@param ["Vertical (Shorts/TikTok)", "Horizontal (YouTube)", "Cuadrado"]
pasos = 30 #@param {type:"slider", min:20, max:45, step:1}

import re, os, zipfile, torch
from IPython.display import display
from PIL import Image

ESTILOS = {
 "Animaci\u00f3n 3D (Pixar)":"Pixar-style 3D animated movie still, cinematic lighting, highly detailed, vibrant colors, soft global illumination, depth of field, charming character design",
 "Anime / Ghibli":"anime style, Studio Ghibli inspired, soft painterly background, cel shading, warm cinematic lighting, detailed",
 "Acuarela infantil":"children book watercolor illustration, soft pastel colors, hand painted, storybook style, cozy",
 "Realista cine":"cinematic photography, ultra realistic, dramatic lighting, 8k, film still, highly detailed",
 "C\u00f3mic":"comic book illustration, bold ink lines, dynamic shading, graphic novel style, vivid colors",
}
DIMS = {"Vertical (Shorts/TikTok)":(832,1216),"Horizontal (YouTube)":(1216,832),"Cuadrado":(1024,1024)}
NEG = "blurry, low quality, deformed, extra limbs, text, watermark, ugly, bad anatomy"

def trocear(t):
    parts = re.split(r"(?<=[\.\!\?\u2026])\s+", re.sub(r"\s+"," ",t).strip())
    out=[]; buf=""
    for p in parts:
        p=p.strip()
        if not p: continue
        if len((buf+" "+p).strip())<45: buf=(buf+" "+p).strip()
        else:
            if buf: out.append(buf)
            buf=p
    if buf: out.append(buf)
    return out[:12]

frases = trocear(historia)
w,h = DIMS[formato]
estilo_p = ESTILOS[estilo]
os.makedirs("/content/escenas", exist_ok=True)
for f in os.listdir("/content/escenas"):
    os.remove("/content/escenas/"+f)

print(f"\U0001F3AC Generando {len(frases)} escenas en formato {formato}...\n")
gen = torch.Generator("cuda")
rutas=[]
for i,fr in enumerate(frases):
    prompt = f"{fr}. Featuring {personaje}. {estilo_p}"
    gen.manual_seed(100+i*7)
    img = pipe(prompt=prompt, negative_prompt=NEG, width=w, height=h,
               num_inference_steps=pasos, guidance_scale=7.0, generator=gen).images[0]
    ruta=f"/content/escenas/escena_{i+1:02d}.png"
    img.save(ruta); rutas.append(ruta)
    print(f"  \u2713 Escena {i+1}: {fr[:60]}...")
    display(img.resize((w//3, h//3)))

zip_path="/content/mis_escenas.zip"
with zipfile.ZipFile(zip_path,"w") as z:
    for r in rutas: z.write(r, os.path.basename(r))
print("\n\u2705 Listo. Descargando el .zip con todas las im\u00e1genes...")
from google.colab import files
files.download(zip_path)


---
### ¿Y ahora qué?
1. Se te descargó **mis_escenas.zip** con las imágenes.
2. Ábrelo y usa esas imágenes en la web **EscenIA** para armar el video con voz, subtítulos y música.
3. ¿Quieres que la web cargue estas imágenes automáticamente? Dímelo y lo conecto.

**Consejo:** si una imagen no te gusta, cambia una palabra de la historia o el estilo y vuelve a ejecutar la última celda.
